In [1]:
import cv2
import os
import numpy as np


frames_folder = "OneDrive/Desktop/Football_Tracking_project/segmentation/foreground_mask"              # Folder containing frames
output_folder = "tracked_frames"      # Folder to save tracked frames

os.makedirs(output_folder, exist_ok=True)

# ==============================
# Load Frame List
# ==============================

frames = sorted(os.listdir(frames_folder))

if len(frames) == 0:
    print("Error: No frames found in folder")
    exit()

# Read the first frame
first_frame_path = os.path.join(frames_folder, frames[0])
frame = cv2.imread(first_frame_path)

if frame is None:
    print("Error: Could not read first frame")
    exit()

print("First frame shape:", frame)

# ==============================
# Select 3 Players
# ==============================

print("Select 3 players to track")

bboxes = []
for i in range(3):
    bbox = cv2.selectROI(f"Select Player {i+1}", frame, False)
    if bbox[2] > 0 and bbox[3] > 0:   # Ensure ROI is valid
        bboxes.append(bbox)
    else:
        print(f"Warning: Invalid ROI selected for Player {i+1}")

cv2.destroyAllWindows()

if len(bboxes) == 0:
    print("Error: No valid bounding boxes selected")
    exit()

# ==============================
# Create Trackers
# ==============================

trackers = []
for bbox in bboxes:
    tracker = cv2.TrackerCSRT_create()
    tracker.init(frame, bbox)
    trackers.append(tracker)

# ==============================
# Process Each Frame
# ==============================

for frame_name in frames:
    frame_path = os.path.join(frames_folder, frame_name)
    frame = cv2.imread(frame_path)

    if frame is None:
        print(f"Warning: Could not read {frame_name}, skipping")
        continue

    # Track each player
    for i, tracker in enumerate(trackers):
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = [int(v) for v in bbox]

            # Draw bounding box
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0,255,0), 2)

            # Add player label
            cv2.putText(frame,
                        f"Player {i+1}",
                        (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,255,0),
                        2)

    # Save tracked frame
    save_path = os.path.join(output_folder, frame_name)
    cv2.imwrite(save_path, frame)

    # Show tracking result
    cv2.imshow("Player Tracking", frame)

    # Press ESC to stop
    if cv2.waitKey(30) & 0xFF == 27:
        break

cv2.destroyAllWindows()


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'OneDrive/Desktop/Football_Tracking_project/segmentation/foreground_mask'